In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import json
from tqdm import tqdm

# Constants
MODEL_PATH = "model/resnet18.pth"
CLASSES_DIR = "model/classes/train"
INPUT_DIR = "output/segmented-characters"

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Load class names
class_names = sorted(
    [d for d in os.listdir(CLASSES_DIR) if os.path.isdir(os.path.join(CLASSES_DIR, d))]
)
num_classes = len(class_names)
print(f"Detected {num_classes} classes.")
print("Sample classes:", class_names[:5])

In [ ]:
# Load model
model = models.resnet18()
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, num_classes)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model = model.to(device)
model.eval()
print("Model loaded successfully.")

In [ ]:
# Image transforms
transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

In [ ]:
def predict_image(image_path):
    img = Image.open(image_path).convert("RGB")
    img_t = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(img_t)
        _, predicted = torch.max(outputs, 1)

    return class_names[predicted.item()]

In [ ]:
# Process characters
predictions = {}

for folder_name in tqdm(os.listdir(INPUT_DIR), desc="Processing folders"):
    folder_base_path = os.path.join(INPUT_DIR, folder_name)
    folder_path = os.path.join(folder_base_path, "middle")
    if not os.path.isdir(folder_path):
        continue

    folder_predictions = []

    images = sorted(
        [f for f in os.listdir(folder_path) if f.lower().endswith((".png"))]
    )
    for img_name in images:
        img_path = os.path.join(folder_path, img_name)
        label = predict_image(img_path)
        folder_predictions.append({"file": img_name, "label": label})

    # Save results
    labels_file_path = os.path.join(folder_base_path, "labels.json")
    with open(labels_file_path, "w") as f:
        json.dump(folder_predictions, f, indent=4)

print("All label files saved in respective folders")